In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
import os
!pip install dagshub mlflow -q
from kaggle_secrets import UserSecretsClient
os.environ["DAGSHUB_USER_TOKEN"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

import dagshub
dagshub.init(repo_owner='lkhiz23', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

import mlflow
print("Connected ✓")

Accessing as lkhiz23

Initialized MLflow to track repo "lkhiz23/IEEE-CIS-Fraud-Detection"

Repository lkhiz23/IEEE-CIS-Fraud-Detection initialized!

Connected ✓


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PATH = '/kaggle/input/competitions/ieee-fraud-detection/'

train_tx = pd.read_csv(PATH + 'train_transaction.csv')
train_id = pd.read_csv(PATH + 'train_identity.csv')
test_tx  = pd.read_csv(PATH + 'test_transaction.csv')
test_id  = pd.read_csv(PATH + 'test_identity.csv')

train = train_tx.merge(train_id, on='TransactionID', how='left')
test  = test_tx.merge(test_id,  on='TransactionID', how='left')

print("Train shape:", train.shape)
print("Test shape: ", test.shape)
train.head()

Train shape: (590540, 434)
Test shape:  (506691, 433)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [6]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Fraud rate in train: {y_train.mean():.4f}")
print(f"Fraud rate in test:  {y_test.mean():.4f}")

X_train: (472432, 432)
X_test:  (118108, 432)
Fraud rate in train: 0.0350
Fraud rate in test:  0.0350


In [7]:
with mlflow.start_run(run_name="DecisionTree_Cleaning"):
    
    # - drop >80% missing cols, calculated on train only -
    missing = X_train.isnull().mean()
    drop_cols = missing[missing > 0.8].index.tolist()
    
    X_train = X_train.drop(columns=drop_cols)
    X_test  = X_test.drop(columns=drop_cols)
    
    print(f"Dropped: {len(drop_cols)} columns")
    print(f"Remaining: {X_train.shape[1]} features")
    
    mlflow.log_param("missing_threshold", 0.8)
    mlflow.log_param("dropped_cols", len(drop_cols))
    mlflow.log_param("remaining_features", X_train.shape[1])

Dropped: 74 | Remaining: 358
🏃 View run DecisionTree_Cleaning at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/a94d07356fe64210bda8c467ea155eb2
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [8]:
with mlflow.start_run(run_name="DecisionTree_Feature_Engineering"):
    
    def feature_engineering(df):
        df = df.copy()
        # - time features -
        df['hour']  = (df['TransactionDT'] // 3600) % 24
        df['day']   = (df['TransactionDT'] // (3600 * 24)) % 7
        df['month'] = (df['TransactionDT'] // (3600 * 24 * 30)) % 12
        # - transaction amount -
        df['log_TransactionAmt'] = np.log1p(df['TransactionAmt'])
        df['cents'] = df['TransactionAmt'] - np.floor(df['TransactionAmt'])
        # - card uid -
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        df['uid2'] = df['uid'] + '_' + df['card3'].astype(str)
        # - email features -
        df['email_match']   = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        df['P_email_count'] = df['P_emaildomain'].map(df['P_emaildomain'].value_counts())
        df['R_email_count'] = df['R_emaildomain'].map(df['R_emaildomain'].value_counts())
        # - frequency encoding -
        df['card1_count'] = df['card1'].map(df['card1'].value_counts())
        df['uid_count']   = df['uid'].map(df['uid'].value_counts())
        # - os and browser -
        if 'id_30' in df.columns:
            df['OS'] = df['id_30'].str.extract(r'^([a-zA-Z\s]+)')
        if 'id_31' in df.columns:
            df['browser'] = df['id_31'].str.extract(r'^([a-zA-Z\s]+)')
        return df
    
    X_train = feature_engineering(X_train)
    X_test  = feature_engineering(X_test)
    
    print(f"Shape after FE: {X_train.shape}")
    mlflow.log_param("added_features", ['hour','day','month','log_TransactionAmt',
                                        'cents','uid','uid2','email_match',
                                        'P_email_count','R_email_count',
                                        'card1_count','uid_count','OS','browser'])
    mlflow.log_param("total_features", X_train.shape[1])

Shape after FE: (472432, 371)
🏃 View run LogisticRegression_Feature_Engineering at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/e8141867e59d4cd0a002d54b28054cf3
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [9]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

with mlflow.start_run(run_name="DecisionTree_Encoding"):
    # - label encode all categorical cols - no scaling needed for trees -
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
        le.fit(combined)
        X_train[col] = le.transform(X_train[col].astype(str))
        X_test[col]  = le.transform(X_test[col].astype(str))
    print(f"Label encoded {len(cat_cols)} columns")
    
    imputer = SimpleImputer(strategy='median')
    X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
    X_test  = pd.DataFrame(imputer.transform(X_test),      columns=X_test.columns)
    print(f"Imputed. Final shape: {X_train.shape}")
    mlflow.log_param("encoding", "LabelEncoder")
    mlflow.log_param("imputation", "median")
    mlflow.log_param("scaling", "None - not needed for trees")

Label encoded 29 columns
Imputed. Final shape: (472432, 371)
🏃 View run DecisionTree_Encoding at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/d1920ca2049e48d8bc8237d892b80424
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectFromModel

with mlflow.start_run(run_name="DecisionTree_Feature_Selection"):
    
    # - method 1: correlation filter (same as before) -
    correlations = pd.Series(
        np.abs(np.corrcoef(X_train.T, y_train)[-1, :-1]),
        index=X_train.columns
    )
    low_corr_cols = correlations[correlations < 0.005].index.tolist()
    X_train_fs = X_train.drop(columns=low_corr_cols)
    X_test_fs  = X_test.drop(columns=low_corr_cols)
    print(f"Correlation filter removed: {len(low_corr_cols)} features")
    
    # - method 2: tree-based feature importance -
    # train shallow tree just for feature selection
    selector_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
    selector_tree.fit(X_train_fs, y_train)
    
    selector = SelectFromModel(selector_tree, prefit=True, threshold='mean')
    selected_cols = X_train_fs.columns[selector.get_support()].tolist()
    
    X_train_fs = X_train_fs[selected_cols]
    X_test_fs  = X_test_fs[selected_cols]
    
    print(f"Tree importance kept: {len(selected_cols)} features")
    print(f"Final shape: {X_train_fs.shape}")
    
    mlflow.log_param("corr_threshold", 0.005)
    mlflow.log_param("corr_removed", len(low_corr_cols))
    mlflow.log_param("selection_method", "DecisionTree_importance")
    mlflow.log_metric("final_features", X_train_fs.shape[1])

Correlation filter removed: 73 features
Tree importance kept: 19 features
Final shape: (472432, 19)
🏃 View run DecisionTree_Feature_Selection at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/0ca60de3d82147f1ba690811b70f2c2b
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [11]:
with mlflow.start_run(run_name="DecisionTree_Feature_Selection_v2_nonzero"):
    
    # - method 1: correlation filter -
    correlations = pd.Series(
        np.abs(np.corrcoef(X_train.T, y_train)[-1, :-1]),
        index=X_train.columns
    )
    low_corr_cols = correlations[correlations < 0.005].index.tolist()
    X_train_fs = X_train.drop(columns=low_corr_cols)
    X_test_fs  = X_test.drop(columns=low_corr_cols)
    print(f"Correlation filter removed: {len(low_corr_cols)} features")
    
    # - method 2: tree importance, keep all nonzero -
    selector_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
    selector_tree.fit(X_train_fs, y_train)
    
    importances = pd.Series(selector_tree.feature_importances_, index=X_train_fs.columns)
    selected_cols = importances[importances > 0].index.tolist()
    
    X_train_fs = X_train_fs[selected_cols]
    X_test_fs  = X_test_fs[selected_cols]
    
    print(f"Tree importance kept: {len(selected_cols)} features")
    print(f"Final shape: {X_train_fs.shape}")
    
    mlflow.log_param("corr_threshold", 0.005)
    mlflow.log_param("corr_removed", len(low_corr_cols))
    mlflow.log_param("selection_method", "tree_importance_nonzero")
    mlflow.log_metric("final_features", X_train_fs.shape[1])

Correlation filter removed: 73 features
Tree importance kept: 25 features
Final shape: (472432, 25)
🏃 View run DecisionTree_Feature_Selection_v2_nonzero at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/637c7930cab34445b80303a8ebc6905c
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0


In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score

# - overfit model - no constraints -
with mlflow.start_run(run_name="DecisionTree_Overfit"):
    model_overfit = DecisionTreeClassifier(random_state=42, class_weight='balanced')
    model_overfit.fit(X_train_fs, y_train)
    
    train_auc = roc_auc_score(y_train, model_overfit.predict_proba(X_train_fs)[:,1])
    test_auc  = roc_auc_score(y_test,  model_overfit.predict_proba(X_test_fs)[:,1])
    
    print(f"Overfit - Train AUC: {train_auc:.4f} | Test AUC: {test_auc:.4f}")
    print(f"Max depth: {model_overfit.get_depth()}")
    
    mlflow.log_param("max_depth", "None")
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc", test_auc)

# - underfit model - too shallow -
with mlflow.start_run(run_name="DecisionTree_Underfit"):
    model_underfit = DecisionTreeClassifier(max_depth=2, random_state=42, class_weight='balanced')
    model_underfit.fit(X_train_fs, y_train)
    
    train_auc = roc_auc_score(y_train, model_underfit.predict_proba(X_train_fs)[:,1])
    test_auc  = roc_auc_score(y_test,  model_underfit.predict_proba(X_test_fs)[:,1])
    
    print(f"Underfit - Train AUC: {train_auc:.4f} | Test AUC: {test_auc:.4f}")
    
    mlflow.log_param("max_depth", 2)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc", test_auc)

# - tuned models -
for depth in [5, 10, 15, 20]:
    with mlflow.start_run(run_name=f"DecisionTree_depth{depth}"):
        model = DecisionTreeClassifier(max_depth=depth, random_state=42, class_weight='balanced')
        
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(model, X_train_fs, y_train, cv=cv, scoring='roc_auc')
        
        model.fit(X_train_fs, y_train)
        y_pred       = model.predict(X_test_fs)
        y_pred_proba = model.predict_proba(X_test_fs)[:,1]
        
        train_auc = roc_auc_score(y_train, model.predict_proba(X_train_fs)[:,1])
        test_auc  = roc_auc_score(y_test, y_pred_proba)
        f1        = f1_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall    = recall_score(y_test, y_pred)
        
        print(f"depth={depth} | Train AUC: {train_auc:.4f} | Test AUC: {test_auc:.4f} | F1: {f1:.4f} | CV: {cv_scores.mean():.4f}")
        
        mlflow.log_param("max_depth", depth)
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("test_auc", test_auc)
        mlflow.log_metric("cv_auc_mean", cv_scores.mean())
        mlflow.log_metric("test_f1", f1)
        mlflow.log_metric("test_precision", precision)
        mlflow.log_metric("test_recall", recall)

Overfit - Train AUC: 0.9977 | Test AUC: 0.6683
Max depth: 53
🏃 View run DecisionTree_Overfit at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/f54507c0b2274ff29d47109267638289
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
Underfit - Train AUC: 0.7158 | Test AUC: 0.7188
🏃 View run DecisionTree_Underfit at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/d3236a4caa874e38b9d278ea08659b4a
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
depth=5 | Train AUC: 0.8085 | Test AUC: 0.8070 | F1: 0.2114 | CV: 0.8076
🏃 View run DecisionTree_depth5 at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/380305f8857f4bb3ad2aa2d1fec52c26
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
depth=10 | Train AUC: 0.8623 | Test AUC: 0.8319 | F1: 0.2489 | CV: 0.8390


In [13]:
with mlflow.start_run(run_name="DecisionTree_Best_Pipeline"):
    best_model = DecisionTreeClassifier(
        max_depth=10,
        random_state=42,
        class_weight='balanced'
    )
    best_model.fit(X_train_fs, y_train)
    
    y_pred       = best_model.predict(X_test_fs)
    y_pred_proba = best_model.predict_proba(X_test_fs)[:,1]
    
    auc       = roc_auc_score(y_test, y_pred_proba)
    f1        = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    train_auc = roc_auc_score(y_train, best_model.predict_proba(X_train_fs)[:,1])
    
    print(f"AUC: {auc:.4f} | F1: {f1:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
    print(classification_report(y_test, y_pred))
    
    mlflow.log_params({"max_depth": 10, "class_weight": "balanced"})
    mlflow.log_metrics({
        "train_auc": train_auc,
        "test_auc": auc,
        "test_f1": f1,
        "test_precision": precision,
        "test_recall": recall
    })
    mlflow.sklearn.log_model(
        best_model,
        "decision_tree_model",
        registered_model_name="DecisionTree_Fraud"
    )
    print(f"Model saved.")

AUC: 0.8319 | F1: 0.2489 | Precision: 0.1519 | Recall: 0.6884
              precision    recall  f1-score   support

           0       0.99      0.86      0.92    113975
           1       0.15      0.69      0.25      4133

    accuracy                           0.85    118108
   macro avg       0.57      0.77      0.58    118108
weighted avg       0.96      0.85      0.90    118108



2026/05/02 22:42:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:42:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'DecisionTree_Fraud'.
2026/05/02 22:42:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DecisionTree_Fraud, version 1
Created version '1' of model 'DecisionTree_Fraud'.


Model saved.
🏃 View run DecisionTree_Best_Pipeline at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0/runs/554c1256a71644348bc10fb1670b22bc
🧪 View experiment at: https://dagshub.com/lkhiz23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/0
